# 04 — Export & Optimization (ONNX / OpenVINO / TensorRT)

Phase 6's exploration notebook, per constitution Principle II: establish *that each export
and quantization path works* before productionizing any of it into
`src/poultry_monitoring/export.py`, `inference.py` and `benchmark.py`.

**The question this phase answers.** Every number in the README so far is PyTorch-on-GPU at
training time. What does this model actually cost to *run* — on a CPU, on this GPU, at FP32
vs FP16 vs INT8 — and what accuracy does each speedup cost?

**What this notebook specifically has to settle**, because the productionized harness is
built on the answers:

1. Do the backends actually work here (Python 3.13, native Windows, Turing GPU, cu124 torch)?
2. Does exporting a `-seg` model preserve its predictions?
3. Does scoring an exported artifact through Ultralytics' validator yield **mask** metrics?
4. Static vs. dynamic input shape — which do the accuracy artifacts need?
5. Which quantization paths run, and which need first-party code?

**Scope.** `yolo26n-seg` (the small checkpoint) throughout — this is a mechanism check, not
a results run. The full matrix across both sizes is the productionized harness's job.

## Setup

### Two things that must happen before `ultralytics` is imported

**Disable AutoUpdate.** When Ultralytics needs a backend it doesn't find, it pip-installs it
mid-run. Loading an ONNX model on CPU calls `check_requirements(("onnx", "onnxruntime"))`
(`nn/backends/onnx.py`), and because this project pins `onnxruntime-gpu`, the *metadata* for
plain `onnxruntime` is absent — so AutoUpdate fetches it. Both packages install into the
same `onnxruntime/` directory, so you end up with two builds fighting over one module, and
whatever it installed is outside `uv.lock` (constitution Principle VII).

This is not hypothetical; it happened while building this notebook. Removing the stray
package then deleted files the GPU build shared, leaving `onnxruntime` importable but
missing `InferenceSession` until it was force-reinstalled.

`AUTOINSTALL` is read at import time from `YOLO_AUTOINSTALL`, so the environment variable
has to be set *before* the first `ultralytics` import.

In [1]:
import os

# Must precede any ultralytics import -- utils/__init__.py reads this once, at import time.
os.environ["YOLO_AUTOINSTALL"] = "false"

from ultralytics.utils import AUTOINSTALL

print(f"Ultralytics AUTOINSTALL: {AUTOINSTALL}  (False = it will not pip-install behind uv.lock)")

Ultralytics AUTOINSTALL: False  (False = it will not pip-install behind uv.lock)


### Import order is load-bearing here

On Windows, ONNX Runtime's CUDA execution provider needs CUDA and cuDNN DLLs on the process
DLL search path. It does **not** find them on its own in this environment — but `torch`
registers its own bundled `lib/` directory when imported, which happens to contain exactly
the CUDA 12 + cuDNN 9 libraries the pinned `onnxruntime-gpu` build wants.

So `import torch` before `onnxruntime` silently decides whether the GPU works. Relying on
that is fragile, so we register the directory explicitly instead — the productionized
`inference.load_model` does the same thing for the same reason.

In [2]:
import warnings
from pathlib import Path

import torch  # before onnxruntime: see above

warnings.filterwarnings("ignore")

# Explicit, rather than depending on torch's import side effect.
TORCH_LIB = Path(torch.__file__).parent / "lib"
os.add_dll_directory(str(TORCH_LIB))

print(f"torch        {torch.__version__}  (built for CUDA {torch.version.cuda})")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    print(f"device       {torch.cuda.get_device_name(0)}  compute capability {cc[0]}.{cc[1]}")
    print("             -> FP16 yes, INT8 yes, BF16/FP8 NO (needs Ampere+/Hopper)")

torch        2.6.0+cu124  (built for CUDA 12.4)
CUDA available: True
device       NVIDIA GeForce RTX 2060 SUPER  compute capability 7.5
             -> FP16 yes, INT8 yes, BF16/FP8 NO (needs Ampere+/Hopper)


### Point at ChickenDet and the checkpoints

In [3]:
DATA_DIR = Path("../data/ChickenDet").resolve()
DATA_YAML = DATA_DIR / "chickendet.yaml"
WEIGHTS = DATA_DIR / "YOLO" / "yolo26n-seg-baseline-adamw" / "weights" / "best.pt"

# Scratch dir for this notebook's artifacts, kept out of the run directories.
SCRATCH = DATA_DIR / "YOLO" / "export" / "_notebook04"
SCRATCH.mkdir(parents=True, exist_ok=True)

assert WEIGHTS.exists(), f"missing checkpoint: {WEIGHTS}"
print(f"checkpoint  {WEIGHTS.relative_to(DATA_DIR)}  ({WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print(f"scratch     {SCRATCH}")

checkpoint  YOLO\yolo26n-seg-baseline-adamw\weights\best.pt  (6.5 MB)
scratch     C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04


### A 60-image evaluation subset

Every accuracy check below runs on 60 images drawn from **Validation**, not Test. Two
reasons: it keeps each `val()` call to seconds so the notebook stays re-runnable, and the
test split stays held out — this notebook is checking mechanisms, so it has no business
touching it (`plan.md`: "test split is off-limits for any decision-making").

The numbers here are therefore *relative* comparisons on a small sample, not results.

In [4]:
from ultralytics.utils import YAML

subset_images = sorted((DATA_DIR / "images" / "Validation").glob("*.jpg"))[:60]
listing = SCRATCH / "subset60.txt"
listing.write_text("\n".join(str(p) for p in subset_images))

SUBSET_YAML = SCRATCH / "subset60.yaml"
YAML.save(
    SUBSET_YAML,
    {
        "path": str(DATA_DIR),
        "train": str(listing),
        "val": str(listing),
        "names": {0: "Chicken"},
    },
)
print(f"{len(subset_images)} validation images -> {SUBSET_YAML.name}")

60 validation images -> subset60.yaml


## 1. Backend availability — verify, don't assume

An installed package is not a working backend. The check that matters is whether a session
actually runs *on the device you asked for*.

This is worth doing carefully because **ONNX Runtime fails soft**: ask for
`CUDAExecutionProvider`, get a session back with no exception raised, and it may have
quietly fallen back to CPU. `get_providers()` is the only thing that tells you. Benchmarking
without this check produces "GPU" numbers that are really CPU numbers.

In [5]:
import numpy as np
import onnxruntime as ort

# A trivial graph is enough to prove a provider initializes.
_tiny = SCRATCH / "tiny.onnx"
if not _tiny.exists():
    class _Tiny(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.c = torch.nn.Conv2d(3, 8, 3, padding=1)

        def forward(self, x):
            return self.c(x).relu()

    torch.onnx.export(_Tiny().eval(), torch.randn(1, 3, 32, 32), str(_tiny), input_names=["images"])

x = np.random.randn(1, 3, 32, 32).astype(np.float32)
for ep in ["CPUExecutionProvider", "CUDAExecutionProvider"]:
    sess = ort.InferenceSession(str(_tiny), providers=[ep])
    got = sess.get_providers()
    sess.run(None, {"images": x})
    print(f"ORT {ep:<26} {'OK' if ep in got else f'SILENTLY FELL BACK to {got}'}")

ORT CPUExecutionProvider       OK


ORT CUDAExecutionProvider      OK


In [6]:
import openvino as ov
import tensorrt as trt

core = ov.Core()
compiled = core.compile_model(core.read_model(str(_tiny)), "CPU")
compiled([x])
print(f"OpenVINO {'CPU':<22} OK   available devices: {core.available_devices}")


# ultralytics/utils/export/engine.py sets these three; TensorRT 11 removed FP16/INT8
# in favour of strongly-typed networks, which is why this project pins tensorrt<11.
needed = ["FP16", "INT8", "GPU_FALLBACK"]
missing = [f for f in needed if not hasattr(trt.BuilderFlag, f)]
print(f"TensorRT {trt.__version__:<22} builder flags missing: {missing or 'none'}")

OpenVINO CPU                    OK   available devices: ['CPU', 'GPU']
TensorRT 10.16.1.11             builder flags missing: none


## 2. Export to ONNX

Two flavours, and the difference turns out to matter more than expected:

- **static** — input shape frozen at export time (`1x3x640x640` here)
- **dynamic** (`dynamic=True`) — batch/height/width left symbolic

One practical note: `model.export()` writes the artifact **next to its source `.pt`**, not
into the working directory. Each variant therefore gets its own directory with its own copy
of the weights, otherwise exports overwrite each other. The productionized `export.py`
handles this placement explicitly.

In [7]:
import shutil

from ultralytics import YOLO


def export_variant(name: str, **kwargs) -> Path:
    """Export into an isolated dir so variants can't overwrite each other's artifacts."""
    d = SCRATCH / name
    d.mkdir(exist_ok=True)
    pt = d / "model.pt"
    if not pt.exists():
        shutil.copy2(WEIGHTS, pt)
    return Path(YOLO(str(pt)).export(**kwargs))


onnx_static = export_variant("onnx_static", format="onnx", imgsz=640, batch=1, device="cpu")
onnx_dynamic = export_variant("onnx_dynamic", format="onnx", imgsz=640, dynamic=True, device="cpu")

for label, p in [("static", onnx_static), ("dynamic", onnx_dynamic)]:
    print(f"{label:<8} {p.name:<12} {p.stat().st_size / 1e6:5.1f} MB")

Ultralytics 8.4.117  Python-3.13.2 torch-2.6.0+cu124 CPU (AMD Ryzen 5 3600XT 6-Core Processor)


YOLO26n-seg summary (fused): 139 layers, 2,689,079 parameters, 0 gradients, 9.1 GFLOPs



PyTorch: starting from 'C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\onnx_static\model.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 300, 38), (1, 32, 160, 160)) (6.2 MB)



ONNX: starting export with onnx 1.22.0 opset 19...


ONNX: slimming with onnxslim 0.1.96...


...


ONNX: starting export with onnx 1.22.0 opset 19...


ONNX: slimming with onnxslim 0.1.96...


ONNX: export success  3.9s, saved as 'C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\onnx_dynamic\model.onnx' (11.7 MB)



Export complete (4.2s)
Results saved to C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\onnx_dynamic\model.onnx
Predict:         yolo predict task=segment model=C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\onnx_dynamic\model.onnx imgsz=640 
Validate:        yolo val task=segment model=C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\onnx_dynamic\model.onnx imgsz=640 data=C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\chickendet.yaml  
Visualize:       https://netron.app


static   model.onnx    11.1 MB
dynamic  model.onnx    12.3 MB


In [8]:
import onnx

for label, p in [("static", onnx_static), ("dynamic", onnx_dynamic)]:
    g = onnx.load(str(p)).graph
    for i in g.input:
        dims = [
            d.dim_param if d.HasField("dim_param") else d.dim_value
            for d in i.type.tensor_type.shape.dim
        ]
        print(f"{label:<8} {i.name}: {dims}")

static   images: [1, 3, 640, 640]
dynamic  images: ['batch', 3, 'height', 'width']


## 3. Does scoring an exported model give mask metrics?

Ultralytics infers a model's task from the file when you don't say. For a `.pt` that works,
because the task is in the checkpoint. For an exported artifact it falls back to **matching
the filename**: `guess_model_task` looks for `-seg` in the file stem or `segment` in the
path parts (`ultralytics/nn/tasks.py`).

Our artifact is `.../yolo26n-seg-baseline-adamw/model.onnx` — the stem is `model`, and no
path component is literally `segment`. So the match fails, and it falls through to
`return "detect"`.

That failure is quiet: you get a working model and a full metrics table, just a
`DetectionValidator` one with no mask columns at all.

In [9]:
def score(model, name, **kwargs):
    """Score a model on the 60-image subset, returning (box_map50, mask_map50 or None)."""
    m = model.val(
        data=str(SUBSET_YAML), split="val", project=str(SCRATCH / "val"), name=name,
        exist_ok=True, verbose=False, plots=False, workers=0, **kwargs,
    )
    seg = getattr(m, "seg", None)
    mask = seg.map50 if seg is not None and getattr(seg, "map50", None) is not None else None
    return m.box.map50, mask


guessed = YOLO(str(onnx_static))
print(f"task guessed from the filename : {guessed.task!r}")
box, mask = score(guessed, "onnx-no-task", batch=1)
print(f"  box_map50={box:.4f}   mask_map50={mask if mask is not None else 'ABSENT'}")

declared = YOLO(str(onnx_static), task="segment")
print(f"\ntask declared explicitly       : {declared.task!r}")
box, mask = score(declared, "onnx-task", batch=1)
print(f"  box_map50={box:.4f}   mask_map50={mask:.4f}")

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.


task guessed from the filename : 'detect'


Ultralytics 8.4.117  Python-3.13.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)


Loading C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\onnx_static\model.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.22.0 with CUDAExecutionProvider


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 96% ━━━━━━━━━━━╸ 58/60 22.3it/s 2.5s<0.1s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 23.0it/s 2.6s

                   all         60       1439      0.932      0.928      0.978      0.874      0.931      0.928      0.977      0.796


Speed: 1.0ms preprocess, 19.1ms inference, 0.0ms loss, 1.2ms postprocess per image


  box_map50=0.9779   mask_map50=0.9772


**So `task="segment"` is mandatory for every exported artifact.** Without it the harness
would silently report detection-only metrics and the entire mask story would vanish. This is
why the productionized code routes *every* model load through a single
`inference.load_model` that always passes `task=`, instead of calling `YOLO(...)` directly
in several places.

## 4. The letterbox trap

Now compare the exported model against its PyTorch source. The static export appears to
*lose* accuracy — before any quantization has happened at all.

In [10]:
pt_box, pt_mask = score(YOLO(str(WEIGHTS)), "pt-default", batch=1)
st_box, st_mask = score(YOLO(str(onnx_static), task="segment"), "onnx-static", batch=1)
dy_box, dy_mask = score(YOLO(str(onnx_dynamic), task="segment"), "onnx-dynamic", batch=1)

print(f"{'model':<26} {'box_map50':>10} {'mask_map50':>11}")
for label, b, m in [
    ("PyTorch .pt", pt_box, pt_mask),
    ("ONNX static (640x640)", st_box, st_mask),
    ("ONNX dynamic", dy_box, dy_mask),
]:
    print(f"{label:<26} {b:>10.4f} {m:>11.4f}")

Ultralytics 8.4.117  Python-3.13.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)


YOLO26n-seg summary (fused): 139 layers, 2,689,079 parameters, 0 gradients, 9.1 GFLOPs


val: Fast image access  (ping: 0.00.0 ms, read: 2994.8757.2 MB/s, size: 682.8 KB)


val: Scanning C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\labels\Validation.cache... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 21.0Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 3% ──────────── 2/60 4.5it/s 0.1s<13.0s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 96% ━━━━━━━━━━━╸ 58/60 23.2it/s 2.6s<0.1s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 22.5it/s 2.7s

                   all         60       1439      0.966      0.935      0.985      0.878      0.972      0.927      0.985      0.802


Speed: 0.8ms preprocess, 21.5ms inference, 0.0ms loss, 1.0ms postprocess per image


model                       box_map50  mask_map50
PyTorch .pt                    0.9849      0.9847
ONNX static (640x640)          0.9779      0.9772
ONNX dynamic                   0.9849      0.9847


The dynamic export matches PyTorch exactly; the static one is consistently below it. It
would be easy to record that as "ONNX export costs ~0.7 mask mAP" and move on.

It doesn't. The hypothesis worth testing: Ultralytics' validator defaults to `rect=True`,
which letterboxes each batch to the image's own aspect ratio (ChickenDet is 1280x720, so
roughly 640x384). A graph frozen at `1x3x640x640` **cannot** do that — it forces square
padding, which is exactly what `rect=False` does.

If that's the cause, then forcing the *PyTorch* model to `rect=False` should reproduce the
static export's numbers precisely.

In [11]:
rect_true = score(YOLO(str(WEIGHTS)), "pt-rect-true", batch=1, rect=True)
rect_false = score(YOLO(str(WEIGHTS)), "pt-rect-false", batch=1, rect=False)

print(f"{'configuration':<34} {'box_map50':>10} {'mask_map50':>11}")
for label, (b, m) in [
    ("PyTorch .pt, rect=True", rect_true),
    ("ONNX dynamic", (dy_box, dy_mask)),
    ("PyTorch .pt, rect=False", rect_false),
    ("ONNX static (640x640)", (st_box, st_mask)),
]:
    print(f"{label:<34} {b:>10.4f} {m:>11.4f}")

Ultralytics 8.4.117  Python-3.13.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)


YOLO26n-seg summary (fused): 139 layers, 2,689,079 parameters, 0 gradients, 9.1 GFLOPs


val: Fast image access  (ping: 0.00.0 ms, read: 2970.6329.4 MB/s, size: 676.2 KB)


val: Scanning C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\labels\Validation.cache... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 21.0Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 5% ╸─────────── 3/60 7.9it/s 0.1s<7.2s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 57/60 24.9it/s 2.3s<0.1s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 24.4it/s 2.5s

                   all         60       1439      0.932      0.928      0.978      0.874      0.931      0.928      0.977      0.796


Speed: 1.1ms preprocess, 16.2ms inference, 0.0ms loss, 1.1ms postprocess per image


configuration                       box_map50  mask_map50
PyTorch .pt, rect=True                 0.9849      0.9847
ONNX dynamic                           0.9849      0.9847
PyTorch .pt, rect=False                0.9779      0.9772
ONNX static (640x640)                  0.9779      0.9772


**Confirmed — the gap is preprocessing, not numerics.** `rect=False` on the PyTorch model
reproduces the static export's numbers exactly, and `rect=True` reproduces the dynamic
export's. ONNX FP32 export is numerically faithful to four decimal places on both box and
mask metrics.

Two consequences the productionized harness has to respect:

1. **A quantization delta is only meaningful against a baseline at the same input shape.**
   Comparing an INT8 static engine against the PyTorch `rect=True` number would bundle a
   fixed letterboxing cost into what gets reported as quantization damage.
2. **Accuracy artifacts and latency artifacts want different exports.** Accuracy scoring
   wants dynamic shapes to stay comparable with the training-time numbers; TensorRT engines
   and INT8 calibration want frozen shapes. These are two exports of the same model, and
   conflating them silently corrupts one measurement or the other.

## 5. Quantization paths

What Ultralytics covers natively, and what needs first-party code.

Worth knowing up front: `int8=True`/`half=True` are **deprecated** in ultralytics 8.4.117 in
favour of a unified `quantize=` argument (`8`, `16`, `32`, `'w8a8'`, `'w8a16'`, `'w8a32'`).
INT8 additionally requires `data=` — a calibration dataset, because static quantization has
to observe real activation ranges to pick its scales.

In [12]:
import time

CASES = [
    ("onnx_fp16", dict(format="onnx", imgsz=640, quantize=16, device=0)),
    ("onnx_int8", dict(format="onnx", imgsz=640, quantize=8,
                       data=str(DATA_YAML), fraction=0.05, device="cpu")),
    ("openvino_fp32", dict(format="openvino", imgsz=640, device="cpu")),
    ("openvino_int8", dict(format="openvino", imgsz=640, quantize=8,
                           data=str(DATA_YAML), fraction=0.05, device="cpu")),
    ("engine_fp16", dict(format="engine", imgsz=640, quantize=16, device=0)),
    ("engine_int8", dict(format="engine", imgsz=640, quantize=8,
                         data=str(DATA_YAML), fraction=0.05, device=0)),
]

quantized = {}
for name, kwargs in CASES:
    t0 = time.time()
    try:
        p = export_variant(name, **kwargs)
        size = (
            sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
            if p.is_dir() else p.stat().st_size
        )
        quantized[name] = p
        print(f"{name:<15} OK   {time.time() - t0:6.1f}s  {size / 1e6:7.1f} MB")
    except Exception as e:
        print(f"{name:<15} FAIL {time.time() - t0:6.1f}s  {type(e).__name__}: {str(e)[:110]}")

Ultralytics 8.4.117  Python-3.13.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)


YOLO26n-seg summary (fused): 139 layers, 2,689,079 parameters, 0 gradients, 9.1 GFLOPs



PyTorch: starting from 'C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\onnx_fp16\model.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 300, 38), (1, 32, 160, 160)) (6.2 MB)



ONNX: starting export with onnx 1.22.0 opset 17...


ONNX: slimming with onnxslim 0.1.96...


...

TensorRT: building INT8 engine as C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\engine_int8\model.engine


TensorRT: keeping 100 Sigmoid layers in FP32 for INT8 accuracy


TensorRT: export success  221.5s, saved as 'C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\engine_int8\model.engine' (6.8 MB)



Export complete (221.7s)
Results saved to C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\engine_int8\model.engine
Predict:         yolo predict task=segment model=C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\engine_int8\model.engine imgsz=640 
Validate:        yolo val task=segment model=C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\YOLO\export\_notebook04\engine_int8\model.engine imgsz=640 data=C:\Users\juane\Documents\Juanes\Proyectos\cv_portfolio\cv_portfolio\poultry_monitoring\data\ChickenDet\chickendet.yaml  
Visualize:       https://netron.app


engine_int8     OK    221.8s      7.1 MB


### The one path Ultralytics does not cover

Dynamic (weight-only) INT8 quantizes weights but leaves activations in FP32, so it needs no
calibration data at all. Ultralytics restricts it to LiteRT — `W8A32_FORMATS =
frozenset({"litert"})` in `engine/exporter.py` — so there is no route to it for ONNX.

It's in the matrix mainly as an instructive comparison: on Conv-heavy graphs it is often
*slower* than FP32 on CPU, because every weight has to be dequantized on the fly. Measuring
that is more useful than asserting it.

In [13]:
from onnxruntime.quantization import quantize_dynamic
from onnxruntime.quantization.quant_utils import QuantType

dyn_int8_path = SCRATCH / "onnx_int8_dynamic" / "model.onnx"
dyn_int8_path.parent.mkdir(exist_ok=True)
quantize_dynamic(str(onnx_dynamic), str(dyn_int8_path), weight_type=QuantType.QInt8)

print(f"FP32 dynamic : {onnx_dynamic.stat().st_size / 1e6:6.1f} MB")
print(f"INT8 dynamic : {dyn_int8_path.stat().st_size / 1e6:6.1f} MB")

FP32 dynamic :   12.3 MB
INT8 dynamic :    4.5 MB


## Summary — what this notebook found

### 1. All four backends work, but only after two version pins
ONNX Runtime (CPU + CUDA EP), OpenVINO and TensorRT all run here. Getting there needed
`onnxruntime-gpu<1.23` (1.23+ links against CUDA 13; torch here is cu124, and the CUDA-12
builds reuse the CUDA 12 + cuDNN 9 DLLs torch already ships) and `tensorrt<11` (TRT 11
removed the `BuilderFlag.FP16`/`INT8` constants that ultralytics 8.4.117 still sets).

### 2. ONNX Runtime's CUDA provider fails silently — and Ultralytics reports it as working
Requesting `CUDAExecutionProvider` returns a working session even when the provider failed
to load; it just runs on CPU, and only `get_providers()` reveals it.

Going through Ultralytics doesn't protect you — it makes it worse. `nn/backends/onnx.py`
gates on `get_available_providers()`, which *lists* CUDA EP even when it cannot load; it
then appends `CPUExecutionProvider` as a fallback and logs "Using ONNX Runtime … with
CUDAExecutionProvider" **before** the session is constructed. Its explicit "CUDA requested
but not available" warning only fires when the provider isn't listed at all — which is not
this failure mode. So the log claims GPU while the work happens on CPU.

The benchmark harness must therefore assert on the session's actual providers after load,
and treat the device argument and the log line as unverified claims.

### 3. Ultralytics will pip-install behind your lockfile
Loading an ONNX model on CPU triggers `check_requirements(("onnx", "onnxruntime"))`, which
AutoUpdate-installs `onnxruntime` alongside the pinned `onnxruntime-gpu`. They share a
module directory, so the two builds collide — and later removing the stray one deleted
shared files, leaving `onnxruntime` importable but without `InferenceSession`. Setting
`YOLO_AUTOINSTALL=false` before importing ultralytics is mandatory, not optional hygiene.

### 4. `task="segment"` is mandatory for exported artifacts
`guess_model_task` matches on the filename, and an exported `model.onnx` doesn't match, so
it silently returns `"detect"` — producing a complete metrics table with no mask columns.

### 5. ONNX export is numerically exact; the apparent loss is letterboxing
A static export scores below its PyTorch source, but forcing the PyTorch model to
`rect=False` reproduces the static number *exactly*, and `rect=True` reproduces the dynamic
export's. The difference is square vs. aspect-ratio padding, not export error. Quantization
deltas must therefore be measured against a baseline at the same input shape, and accuracy
artifacts (dynamic) are a different export from latency artifacts (frozen shape).

### 6. Ultralytics covers every quantization path except dynamic INT8
FP16 and static INT8 are native via `quantize=`, including a real calibration reader that
deliberately excludes non-`Conv`/`Gemm`/`MatMul` nodes so the detection head stays float —
one INT8 scale spanning box pixels (~0-640) and class probabilities (0-1) would round every
score to zero. Weight-only INT8 is LiteRT-only, so ONNX dynamic INT8 is the single path
needing first-party code.

### Next step
Productionize into `src/poultry_monitoring/`: `export.py` (matrix + artifact manifest),
`inference.py` (the single model-load path, owning `task=` and the DLL registration),
and `benchmark.py` (latency under constitution Principle V). Accuracy degradation is
measured by extending `segmentation/evaluation.py`, which already does held-out and
density-stratified scoring.